# 05 — Fine-tune YourMT3+ on the Restrudel corpus

Goal 2: **fine-tune the released YourMT3+ checkpoint** so it transcribes
synth-heavy electronic music. Structure:

1. **Imports + Drive** — everything import-y, Drive mount, paths.
2. **Configuration** — every knob (incl. the `SMOKE_TEST` switch).
3. **Base model (code)** — fetch YourMT3 code, Drive-cached; patch a loss logger.
4. **Preprocessing / indexing** — rebase paths, per-category test lists, register presets.
5. **Load data** — `x_/y_` variables per category (audio paths = X, label paths = Y).
6. **Split composition** — pie charts of the fine-tune / val / test mixes.
7. **Fine-tune** — load checkpoint (Drive-cached), run (smoke or real), save to Drive + metadata.
8. **Training curves** — train/val loss.
9. **Benchmark** — fine-tuned vs base YourMT3+, per category.

> Prereq: datasets already in Drive (notebook 04). This notebook reads their index files.


## 1. Imports, Google Drive, environment


In [ ]:
# Everything import-y lives here.
import json, os, re, sys, csv, time, shutil, subprocess, datetime
from pathlib import Path
from collections import defaultdict, OrderedDict

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/restrudel")
    if not Path("/content/restrudel").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/henrik253/Restrudel.git", "/content/restrudel"], check=True)
    REPO = Path("/content/restrudel")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
else:
    REPO = Path(os.environ.get("RESTRUDEL_REPO", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
    DRIVE = Path(os.environ.get("RESTRUDEL_DRIVE", REPO))

DATA_HOME = DRIVE / "datasets"
INDEX_DIR = DATA_HOME / "yourmt3_indexes"
MODEL_ROOT = REPO / "models" / "YourMT3"            # cwd for train.py / test.py
AMT_SRC = MODEL_ROOT / "amt" / "src"
DRIVE_MODEL_CACHE = DRIVE / "models" / "YourMT3"    # persistent base-model cache

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
try:
    import torch
    GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
except Exception:
    torch, GPU = None, None

print(f"colab={IN_COLAB}  repo={REPO}")
print(f"data_home={DATA_HOME}")
print(f"GPU={GPU}")
assert INDEX_DIR.exists(), "no index dir — build the datasets in Drive (notebook 04) first"


## 2. Configuration

Every knob in one place. `SMOKE_TEST=True` runs a tiny subset + few steps to
prove the whole pipeline runs end-to-end before committing to a real run.


In [ ]:
TIMESTAMP = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# --- categories: strudel is the TARGET (split into sub-cats); rest are references
SEED2BATCH = {"1": "batch_1", "1000": "batch_2", "2000": "batch_3", "3000": "batch_4", "4000": "batch_5"}
CATEGORIES = OrderedDict([
    ("strudel_corpus",            ("Strudel · corpus",   "target",    "electronic", "#d1495b")),
    ("strudel_synthetic_batch_1", ("Strudel · synth b1", "target",    "electronic", "#e36414")),
    ("strudel_synthetic_batch_2", ("Strudel · synth b2", "target",    "electronic", "#e8871e")),
    ("strudel_synthetic_batch_3", ("Strudel · synth b3", "target",    "electronic", "#f3a712")),
    ("strudel_synthetic_batch_4", ("Strudel · synth b4", "target",    "electronic", "#f7c548")),
    ("maestro",                   ("MAESTRO (piano)",    "reference", "acoustic",   "#66a182")),
    ("slakh",                     ("Slakh (band)",       "reference", "acoustic",   "#2e4057")),
    ("egmd",                      ("EGMD (e-drums)",     "reference", "electronic", "#8d99ae")),
])
STRUDEL_SUBCATS = [c for c in CATEGORIES if c.startswith("strudel")]

def strudel_subcat(sid):
    if sid.startswith("corpus_"):
        return "strudel_corpus"
    m = re.match(r"inspired_(\d+)_\d+$", sid)          # inspired_<seed>_<i> (batch 2+)
    if m:
        return "strudel_synthetic_" + SEED2BATCH.get(m.group(1), "seed" + m.group(1))
    if re.match(r"inspired_\d+$", sid):                # inspired_<NNN> (seed 1 = batch 1)
        return "strudel_synthetic_batch_1"
    return "strudel_corpus"

DIRNAME = {"slakh": "slakh2100_yourmt3_16k", "maestro": "maestro_yourmt3_16k", "egmd": "egmd_yourmt3_16k"}

# --- TWO fine-tuning weight variants to train & compare ----------------------
WEIGHT_VARIANTS = OrderedDict([
    ("strudel50", OrderedDict([("strudel", 0.50), ("slakh", 0.25), ("maestro", 0.15), ("egmd", 0.10)])),
    ("egmd50",    OrderedDict([("strudel", 0.10), ("slakh", 0.25), ("maestro", 0.15), ("egmd", 0.50)])),
])
for _w in WEIGHT_VARIANTS.values():
    assert abs(sum(_w.values()) - 1.0) < 1e-9
WEIGHTS = WEIGHT_VARIANTS["strudel50"]     # §6 composition pie shows variant A's nominal mix

VARIANT_EXP     = {v: f"ft_{v}_{TIMESTAMP}" for v in WEIGHT_VARIANTS}
VARIANT_RUN_DIR = {v: DRIVE / "checkpoints" / f"YourMT3+_fine_tuned_{v}_{TIMESTAMP}" for v in WEIGHT_VARIANTS}
COMPARE_DIR = DRIVE / "checkpoints" / f"comparison_{TIMESTAMP}"   # shared comparison artifacts
for _d in [*VARIANT_RUN_DIR.values(), COMPARE_DIR]:
    _d.mkdir(parents=True, exist_ok=True)
RUN_DIR  = COMPARE_DIR                      # older cells (composition) save here
RUN_NAME = f"comparison_{TIMESTAMP}"

# --- smoke test vs real run --------------------------------------------------
SMOKE_TEST = False            # <<< REAL RUN: 2 variants x 3000 steps
MAX_STEPS  = 10 if SMOKE_TEST else 3000
_VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9 if (torch and torch.cuda.is_available()) else 0
BATCH_SIZE = (1, 2) if SMOKE_TEST else ((8, 16) if _VRAM_GB >= 70 else (4, 8) if _VRAM_GB >= 38 else (2, 4))
LR         = 1e-4
FULL_EVAL  = False            # False = cap each category's test split; True = whole split
EVAL_CAP   = 8 if SMOKE_TEST else 200

PRECISION = "bf16-mixed" if (torch and torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else "16-mixed"

# architecture flags — MUST match the released checkpoint (scripts/yourmt3_transcribe.py)
ARCH = ["-tk", "mc13_full_plus_256", "-dec", "multi-t5", "-nl", "26",
        "-enc", "perceiver-tf", "-sqr", "1", "-ff", "moe", "-wf", "4",
        "-nmoe", "8", "-kmoe", "2", "-act", "silu", "-epe", "rope", "-rp", "1",
        "-ac", "spec", "-hop", "300", "-atc", "1"]
RELEASED_EXP = "mc13_256_g4_all_v7_mt3f_sqr_rms_moe_wf4_n8k2_silu_rope_rp_b36_nops"

print(f"comparison run {TIMESTAMP}: variants={list(WEIGHT_VARIANTS)}")
print(f"smoke={SMOKE_TEST}  max_steps={MAX_STEPS}  bsz={BATCH_SIZE}  precision={PRECISION}  eval_cap={EVAL_CAP}")


## 3. Base model — code + training deps (Drive-cached)

Fetch YourMT3's `amt/src` from the HF **Space** (`repo_type="space"`), restoring
from the Drive cache if already downloaded. Then install YourMT3's training
dependencies and apply the compatibility/memory **source patches** needed to run
it on current Colab (torch ≥2.6, no wandb, 40 GB GPU, data on Drive):

- deps: `pytorch-lightning`, **`transformers==4.39.3`** (4.40+ removed
  `model_parallel_utils`), a `mir_eval` fork + cosine-annealing scheduler;
- `config.py`: `data_home` → our Drive datasets, `num_workers`→1 (Drive FUSE);
- a `CSVLogger` (the Trainer ships with none) for the loss curves;
- guard `wandb_logger` (None when wandb is off) and `torch.load(weights_only=False)`;
- disable `torch.compile` + enable **gradient checkpointing** on the T5 stacks
  (fits the MoE decoder in 40 GB), with the matching decoder-return fix.

All patches are idempotent (marker-guarded). The heavy checkpoint is fetched in §7.


In [ ]:
from huggingface_hub import snapshot_download

def ensure_code():
    if (AMT_SRC / "config" / "data_presets.py").exists():
        print("model code already local:", AMT_SRC); return
    cache_src = DRIVE_MODEL_CACHE / "amt" / "src"
    if (cache_src / "config" / "data_presets.py").exists():
        print("restoring model code from Drive cache…")
        shutil.copytree(DRIVE_MODEL_CACHE / "amt" / "src", AMT_SRC, dirs_exist_ok=True)
    else:
        print("fetching model code from HF Space (one-time)…")
        snapshot_download(repo_id="mimbres/YourMT3", repo_type="space",
                          allow_patterns=["amt/src/**", "amt/src/*"], local_dir=str(MODEL_ROOT))
        (DRIVE_MODEL_CACHE / "amt").mkdir(parents=True, exist_ok=True)
        shutil.copytree(AMT_SRC, DRIVE_MODEL_CACHE / "amt" / "src", dirs_exist_ok=True)
    assert (AMT_SRC / "config" / "data_presets.py").exists(), "model code missing after ensure_code()"
    print("model code ready:", AMT_SRC)

ensure_code()

# --- YourMT3 training deps. transformers PINNED <4.40 (newer removed
#     transformers.utils.model_parallel_utils, imported by the T5 wrapper).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pytorch-lightning>=2.2.1", "transformers==4.39.3", "mirdata", "mido",
                "deprecated", "smart-open", "einops", "wandb",
                "git+https://github.com/craffel/mir_eval.git",
                "git+https://github.com/katsura-jp/pytorch-cosine-annealing-with-warmup.git"], check=True)
print("installed YourMT3 training deps")

# --- config.py: data_home -> our Drive datasets (default ../../data); num_workers=1
#     (Google Drive FUSE throws Errno 5 under multiple parallel dataloader workers).
cfgpy = AMT_SRC / "config" / "config.py"
_cfg = cfgpy.read_text()
_cfg = re.sub(r'"data_home":\s*"[^"]*"', f'"data_home": "{DATA_HOME}"', _cfg, count=1)
_cfg = re.sub(r'"num_workers":\s*\d+', '"num_workers": 1', _cfg, count=1)
cfgpy.write_text(_cfg)
print("config.py: data_home ->", DATA_HOME, "| num_workers -> 1")

# --- YourMT3 source compat patches (idempotent; marker-guarded) --------------
def _patch(relpath, marker, old, new, note):
    p = AMT_SRC / relpath
    t = p.read_text()
    if marker in t:
        print(f"  {note}: already applied"); return
    assert old in t, f"anchor not found for '{note}' in {relpath} (YourMT3 source changed?)"
    p.write_text(t.replace(old, new, 1)); print(f"  {note}: patched")

# (1) attach a CSVLogger (Trainer ships with none) -> loss curves in §8
_patch("model/init_train.py", "restrudel CSVLogger",
       "    trainer = pl.trainer.trainer.Trainer(**train_params)",
       '    from pytorch_lightning.loggers import CSVLogger as _RS_CSV  # restrudel CSVLogger\n'
       '    train_params["logger"] = _RS_CSV(save_dir=lightning_dir, name="metrics")\n'
       "    trainer = pl.trainer.trainer.Trainer(**train_params)",
       "CSVLogger")

# (1b) log EVERY step + validate mid-run — otherwise short (smoke) runs produce an
#      empty loss curve: PL's default flushes train_loss every 50 steps and runs
#      validation only at epoch end, which a 10-step run never reaches.
_patch("model/init_train.py", "restrudel log-cadence",
       '    train_params["logger"] = _RS_CSV(save_dir=lightning_dir, name="metrics")',
       '    train_params["logger"] = _RS_CSV(save_dir=lightning_dir, name="metrics")\n'
       '    train_params["log_every_n_steps"] = 1  # restrudel log-cadence\n'
       '    if int(getattr(args, "max_steps", -1) or -1) > 0:\n'
       '        train_params["val_check_interval"] = max(1, int(args.max_steps) // 2)',
       "log cadence")

# (2) guard wandb_logger (None when wandb disabled)
_wp = AMT_SRC / "train.py"
_wt = _wp.read_text()
if "restrudel-wandb-guard" not in _wt:
    _wt = _wt.replace(
        "    if trainer.global_rank == 0:\n        wandb_logger.experiment.config.update",
        "    if trainer.global_rank == 0 and wandb_logger is not None:  # restrudel-wandb-guard\n        wandb_logger.experiment.config.update", 1)
    _wt = _wt.replace(
        "    wandb_logger.watch(model, log='gradients', log_freq=5000)",
        "    if wandb_logger is not None:\n        wandb_logger.watch(model, log='gradients', log_freq=5000)", 1)
    _wp.write_text(_wt); print("  wandb guard: patched")
else:
    print("  wandb guard: already applied")

# (3) torch>=2.6 defaults torch.load(weights_only=True); the official ckpt is trusted
for _f in ("train.py", "test.py"):
    _p = AMT_SRC / _f
    _t = _p.read_text()
    if 'torch.load(dir_info["last_ckpt_path"])' in _t:
        _p.write_text(_t.replace('torch.load(dir_info["last_ckpt_path"])',
                                 'torch.load(dir_info["last_ckpt_path"], weights_only=False)'))
        print(f"  torch.load weights_only: patched {_f}")

# (4) disable torch.compile in the Perceiver (inductor is memory-hungry -> OOM)
_patch("model/perceiver_mod.py", "restrudel-no-compile",
       "    def forward(self, **kwargs):\n        if self.training is True:\n            return self._forward_compile(**kwargs)\n        else:\n            return self._forward_no_compile(**kwargs)",
       "    def forward(self, **kwargs):  # restrudel-no-compile\n        return self._forward_no_compile(**kwargs)",
       "torch.compile off")

# (5) gradient checkpointing on the T5 stacks (default off) -> fits decoder in 40GB
_patch("model/t5mod.py", "restrudel-grad-ckpt",
       "        self.gradient_checkpointing = False",
       "        self.gradient_checkpointing = True  # restrudel-grad-ckpt",
       "gradient checkpointing")

# (6) with grad-ckpt use_cache=False, the decoder returns a 1-tuple -> take [0]
_patch("model/ymt3.py", "restrudel-dec-unpack",
       "        dec_hs, _ = self.decoder(inputs_embeds=dec_inputs_embeds, encoder_hidden_states=enc_hs, return_dict=False)",
       "        dec_hs = self.decoder(inputs_embeds=dec_inputs_embeds, encoder_hidden_states=enc_hs, return_dict=False)[0]  # restrudel-dec-unpack",
       "decoder unpack")
# (7) explicitly save final weights after fit: Lightning's ModelCheckpoint monitors
#     validation/macro_onset_f (never logged by our runs) so it silently never fires —
#     without this, last.ckpt stays the SEED and the "fine-tuned" model equals the base.
_patch("train.py", "restrudel-save-final",
       '        trainer.fit(model, ckpt_path=dir_info["last_ckpt_path"], datamodule=dm)',
       '        trainer.fit(model, ckpt_path=dir_info["last_ckpt_path"], datamodule=dm)\n\n'
       '    trainer.save_checkpoint(dir_info["lightning_dir"] + "/checkpoints/last.ckpt")  # restrudel-save-final',
       "final save")
print("compat patches applied.")


## 4. Preprocessing & indexing

(a) **Rebase** the strudel index paths onto this `DATA_HOME` (they were written
with absolute build-machine paths). (b) Build **per-category test file-lists**
for benchmarking (non-destructive; capped unless `FULL_EVAL`). (c) Register the
`strudel` + `strudel_ft` training presets and a `bench_<cat>` eval preset per
category in `data_presets.py` (idempotent).


In [ ]:
# (a) rebase strudel index paths onto this DATA_HOME
SONG_ROOT = DATA_HOME / "strudel_yourmt3_16k"
for split in ("train", "validation", "test"):
    p = INDEX_DIR / f"strudel_{split}_file_list.json"
    fl = json.load(open(p)); n = 0
    for e in fl.values():
        sid = e.get("strudel_id")
        if not sid:
            continue
        d = SONG_ROOT / sid
        for key, fname in (("mix_audio_file", "mix.wav"), ("note_events_file", f"{sid}_note_events.npy"),
                           ("notes_file", f"{sid}_notes.npy"), ("midi_file", f"{sid}.mid")):
            if key in e and e[key] != str(d / fname):
                e[key] = str(d / fname); n += 1
    json.dump(fl, open(p, "w"), indent=4)
    print(f"rebased strudel {split}: {n} paths")

# (b) per-category benchmark test lists (capped unless FULL_EVAL)
def load_fl(path):
    return json.load(open(path)) if path.exists() else {}

bench_entries = defaultdict(list)
for e in load_fl(INDEX_DIR / "strudel_test_file_list.json").values():
    bench_entries[strudel_subcat(e["strudel_id"])].append(e)
for cat in ("maestro", "slakh", "egmd"):
    for e in load_fl(INDEX_DIR / f"{cat}_test_file_list.json").values():
        bench_entries[cat].append(e)

BENCH_N = {}
for cat in CATEGORIES:
    entries = bench_entries.get(cat, [])
    use = entries if FULL_EVAL else entries[:EVAL_CAP]
    json.dump({str(i): e for i, e in enumerate(use)},
              open(INDEX_DIR / f"bench_{cat}_test_file_list.json", "w"), indent=4)
    BENCH_N[cat] = len(use)
print("bench test lists (files/category):", dict(BENCH_N))

# (c) register presets (idempotent)
DP = AMT_SRC / "config" / "data_presets.py"
# cat -> (eval_vocab_expr, eval_drum_vocab_expr, has_stem)
VOCAB = {"maestro": ("[PIANO_SOLO_CLASS]", None, False),
         "slakh":   ("[GM_INSTR_CLASS]", 'drum_vocab_presets["gm"]', True),
         "egmd":    ("[None]", 'drum_vocab_presets["ksh"]', False)}
def strudel_vocab():
    return ("[GM_INSTR_CLASS]", 'drum_vocab_presets["gm"]', False)

def single_preset(name, dataset_name, ev, edv, has_stem, splits=("train", "validation", "test")):
    L = [f'    "{name}": {{', f'        "eval_vocab": {ev},']
    if edv:
        L.append(f'        "eval_drum_vocab": {edv},')
    L += [f'        "dataset_name": "{dataset_name}",',
          f'        "train_split": "{splits[0]}",',
          f'        "validation_split": "{splits[1]}",',
          f'        "test_split": "{splits[2]}",',
          f'        "has_stem": {has_stem},', '    },']
    return "\n".join(L)

src = DP.read_text()
if "restrudel presets v2" not in src:
    singles = ["    # >>> restrudel presets v2 >>>",
               single_preset("strudel", "strudel", "[GM_INSTR_CLASS]", 'drum_vocab_presets["gm"]', False)]
    for cat in CATEGORIES:                              # bench_<cat>: all splits point at the test list
        ev, edv, hs = strudel_vocab() if cat.startswith("strudel") else VOCAB[cat]
        singles.append(single_preset(f"bench_{cat}", f"bench_{cat}", ev, edv, hs,
                                     splits=("test", "test", "test")))
    singles.append("    # <<< restrudel presets v2 <<<")
    single_block = "\n".join(singles) + "\n"
    multi_block = "\n".join([
        "    # >>> restrudel strudel_ft v2 >>>",
        '    "strudel_ft": {',
        '        "presets": ["strudel", "slakh", "maestro", "egmd"],',
        f'        "weights": {list(WEIGHTS.values())},',
        "        \"eval_vocab\": [GM_INSTR_CLASS, None, None, None],",
        '        "eval_drum_vocab": drum_vocab_presets["gm"],',
        '        "val_max_num_files": 20,',
        '        "test_max_num_files": None,',
        "    },",
        "    # <<< restrudel strudel_ft v2 <<<"]) + "\n"
    src = src.replace("data_preset_single_cfg = {\n", "data_preset_single_cfg = {\n" + single_block, 1)
    src = src.replace("data_preset_multi_cfg = {\n", "data_preset_multi_cfg = {\n" + multi_block, 1)
    DP.write_text(src)
    print("registered strudel + strudel_ft + bench presets")
else:
    print("presets already registered")

need = ["strudel", "strudel_ft"] + [f"bench_{c}" for c in CATEGORIES]
have = DP.read_text()
for nm in need:
    assert f'"{nm}"' in have, f"preset {nm} missing"
print("presets present:", need)

# (d) register the two weight-variant multi presets (v3)
src = DP.read_text()
if "restrudel ft variants v3" not in src:
    blocks = ["    # >>> restrudel ft variants v3 >>>"]
    for vname, w in WEIGHT_VARIANTS.items():
        blocks += [f'    "strudel_ft_{vname}": {{',
                   '        "presets": ["strudel", "slakh", "maestro", "egmd"],',
                   f'        "weights": {[w["strudel"], w["slakh"], w["maestro"], w["egmd"]]},',
                   '        "eval_vocab": [GM_INSTR_CLASS, None, None, None],',
                   '        "eval_drum_vocab": drum_vocab_presets["gm"],',
                   '        "val_max_num_files": 20,',
                   '        "test_max_num_files": None,',
                   "    },"]
    blocks.append("    # <<< restrudel ft variants v3 <<<")
    src = src.replace("data_preset_multi_cfg = {\n", "data_preset_multi_cfg = {\n" + "\n".join(blocks) + "\n", 1)
    DP.write_text(src)
    print("registered ft variant presets:", [f"strudel_ft_{v}" for v in WEIGHT_VARIANTS])
else:
    print("ft variant presets already registered")
for vname in WEIGHT_VARIANTS:
    assert f'"strudel_ft_{vname}"' in DP.read_text(), f"variant preset {vname} missing"


## 5. Load data — `x_/y_` variables per category

YourMT3 trains from **file lists with lazy loading** (the corpus is ~770 h /
100s of GB decoded), so `x_*` hold **audio-file paths** and `y_*` hold the
matching **label (note-event) paths** — the faithful representation at this
scale. Everything is in the `DATA[split][cat]` dict *and* exposed as flat
variables like `x_train_strudel_corpus`, `y_test_maestro`, etc.


In [ ]:
SPLITS = ("train", "validation", "test")
DATA  = {s: {c: {"x": [], "y": []} for c in CATEGORIES} for s in SPLITS}
HOURS = {s: {c: 0.0 for c in CATEGORIES} for s in SPLITS}

def _xy(e):
    return e.get("mix_audio_file") or e.get("stem_file"), e.get("note_events_file")

for split in SPLITS:
    # strudel -> bucket by sub-category
    for e in load_fl(INDEX_DIR / f"strudel_{split}_file_list.json").values():
        cat = strudel_subcat(e["strudel_id"]); x, y = _xy(e)
        DATA[split][cat]["x"].append(x); DATA[split][cat]["y"].append(y)
        HOURS[split][cat] += e.get("n_frames", 0) / 16000 / 3600
    # references
    for cat in ("maestro", "slakh", "egmd"):
        for e in load_fl(INDEX_DIR / f"{cat}_{split}_file_list.json").values():
            x, y = _xy(e)
            DATA[split][cat]["x"].append(x); DATA[split][cat]["y"].append(y)
            HOURS[split][cat] += e.get("n_frames", 0) / 16000 / 3600

# expose the flat x_/y_ variables the design asks for
for split in SPLITS:
    for cat in CATEGORIES:
        globals()[f"x_{split}_{cat}"] = DATA[split][cat]["x"]
        globals()[f"y_{split}_{cat}"] = DATA[split][cat]["y"]

print(f"{'category':<28}{'train':>7}{'val':>6}{'test':>6}{'hours':>8}")
print("-" * 55)
for cat in CATEGORIES:
    tr, va, te = (len(DATA[s][cat]['x']) for s in SPLITS)
    print(f"{cat:<28}{tr:>7}{va:>6}{te:>6}{sum(HOURS[s][cat] for s in SPLITS):>8.1f}")
print("-" * 55)
print(f"{'TOTAL':<28}" + "".join(f"{sum(len(DATA[s][c]['x']) for c in CATEGORIES):>{w}}"
      for s, w in zip(SPLITS, (7, 6, 6))) + f"{sum(HOURS[s][c] for s in SPLITS for c in CATEGORIES):>8.1f}")
print("example:  x_train_strudel_corpus[0] =", (x_train_strudel_corpus[0] if x_train_strudel_corpus else None))


## 6. Split composition

Three views of the corpus — **read the units carefully, they differ per pie**:

- **Left — fine-tuning mix.** Wedges are **sampling probabilities** (draw %), *not*
  song counts: how often each category is drawn into a training batch (with
  replacement). They sum to 100 %.
- **Middle / right — validation & test.** Wedges are **physical song counts** per
  category.

> ⚠️ **Don't multiply the left pie by the pool size.** The train **pool** is ~40k
> songs but is **mostly reference data** (EGMD/Slakh); Strudel is only ~2.2k train
> songs (451 real corpus, the rest synthetic batches 1–4). So *Strudel·corpus ≈
> 10 %* is a **draw probability** (0.50 × 451/2205), **not** ~4k songs. The raw
> weight-vs-count breakdown is printed beneath the plot.


In [ ]:
# per-split totals (songs + hours). The train pool is dominated by the big reference
# sets (EGMD/Slakh) — Strudel is a small fraction; see the raw breakdown printed below.
TOT_SONGS = {s: sum(len(DATA[s][c]["x"]) for c in CATEGORIES) for s in SPLITS}
TOT_HOURS = {s: sum(HOURS[s][c] for c in CATEGORIES) for s in SPLITS}

# fine-tuning mix: each slice's DRAW PROBABILITY (sampling weight) + its underlying song pool
strudel_train = {c: len(DATA["train"][c]["x"]) for c in STRUDEL_SUBCATS}
s_tot = sum(strudel_train.values()) or 1
mix = []
for c in STRUDEL_SUBCATS:                       # Strudel's 0.50 split by each sub-cat's share
    mix.append({"label": CATEGORIES[c][0], "weight": WEIGHTS["strudel"] * strudel_train[c] / s_tot,
                "songs": len(DATA["train"][c]["x"]), "color": CATEGORIES[c][3]})
for c in ("slakh", "maestro", "egmd"):
    mix.append({"label": CATEGORIES[c][0], "weight": WEIGHTS[c],
                "songs": len(DATA["train"][c]["x"]), "color": CATEGORIES[c][3]})

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 5.4))

# (1) fine-tuning mix — wedges are SAMPLING PROBABILITIES (draw %), NOT song counts
ax1.pie([m["weight"] for m in mix], labels=[m["label"] for m in mix], colors=[m["color"] for m in mix],
        startangle=90, counterclock=False, autopct=lambda p: f"{p:.0f}%" if p >= 6 else "",
        textprops={"fontsize": 8}, wedgeprops=dict(width=0.45, edgecolor="white"))
ax1.set_title("fine-tuning sampling mix\n(wedge = draw probability, not count)", fontsize=10)

# (2) validation, (3) test — these ARE physical song counts per category
for ax, split, title in ((ax2, "validation", "validation set"), (ax3, "test", "test set")):
    vals = [len(DATA[split][c]["x"]) for c in CATEGORIES]
    ax.pie(vals, colors=[CATEGORIES[c][3] for c in CATEGORIES], startangle=90, counterclock=False,
           autopct=lambda p: f"{p:.0f}%" if p >= 6 else "", textprops={"fontsize": 8},
           wedgeprops=dict(width=0.45, edgecolor="white"))
    ax.set_title(f"{title} — song counts\n{TOT_SONGS[split]:,} songs · {TOT_HOURS[split]:.0f} h", fontsize=10)

fig.legend([CATEGORIES[c][0] for c in CATEGORIES], loc="lower center", ncol=4, frameon=False, fontsize=8)
fig.suptitle("Split composition — fine-tune mix (probabilities) vs. validation / test (counts)",
             fontsize=12, y=1.03)

# caption baked into the figure so the saved PNG is self-explanatory
corpus_tr = len(DATA["train"]["strudel_corpus"]["x"]); strudel_tr = sum(strudel_train.values()) or 1
cap = (f"LEFT = draw probability per category (sampling weights, sum 100%), NOT counts.   "
       f"Train pool = {TOT_SONGS['train']:,} songs / {TOT_HOURS['train']:.0f} h, mostly reference sets; "
       f"Strudel is only {strudel_tr:,} train songs ({corpus_tr} real corpus).\n"
       f"So 'Strudel-corpus {100 * 0.50 * corpus_tr / strudel_tr:.0f}%' is a draw probability "
       f"(0.50 x {corpus_tr}/{strudel_tr}), not ~{0.50 * TOT_SONGS['train']:,.0f} songs (the naive 50% x pool).")
fig.text(0.5, -0.02, cap, ha="center", va="top", fontsize=8,
         bbox=dict(boxstyle="round", fc="#f7f7f7", ec="#cccccc"))

plt.tight_layout(rect=[0, 0.10, 1, 1])
fig.savefig(RUN_DIR / "composition.png", dpi=120, bbox_inches="tight"); plt.show()
print("saved ->", RUN_DIR / "composition.png")

# ---- raw breakdown beneath the plot (resolves the weight-vs-count confusion) ----
print("\nFINE-TUNING MIX — sampling weight (draw probability)  vs.  underlying train pool:")
for m in mix:
    print(f"  {m['label']:<20} {m['weight'] * 100:5.1f}% draw  ·  {m['songs']:>6,} train songs")
print(f"  {'-' * 48}")
print(f"  {'TOTAL':<20} {sum(m['weight'] for m in mix) * 100:5.1f}% (weights)  ·  "
      f"{sum(m['songs'] for m in mix):>6,} songs (physical pool)")

print(f"\nPHYSICAL POOL — song counts x split (where the ~{TOT_SONGS['train']:,} train figure comes from):")
tbl = pd.DataFrame(
    [{"category": CATEGORIES[c][0],
      "train": len(DATA["train"][c]["x"]), "val": len(DATA["validation"][c]["x"]),
      "test": len(DATA["test"][c]["x"]), "hours": round(sum(HOURS[s][c] for s in SPLITS), 1)}
     for c in CATEGORIES]).set_index("category")
tbl.loc["TOTAL"] = [tbl["train"].sum(), tbl["val"].sum(), tbl["test"].sum(), round(tbl["hours"].sum(), 1)]
tbl[["train", "val", "test"]] = tbl[["train", "val", "test"]].astype(int)  # counts as ints, not 451.0
try:
    display(tbl)
except NameError:
    print(tbl.to_string())

print(f"\nNote: LEFT-pie wedges are draw probabilities (with-replacement sampling) — a category's slice "
      f"is NOT its song count. The real corpus ({corpus_tr} train / "
      f"{len(DATA['validation']['strudel_corpus']['x'])} val / {len(DATA['test']['strudel_corpus']['x'])} test) "
      f"is drawn ~{100 * 0.50 * corpus_tr / strudel_tr:.0f}% of the time; the {TOT_SONGS['train']:,}-song pool "
      f"is mostly reference data (EGMD/Slakh), not Strudel.")


## 7. Fine-tune — two weight variants, background driver

Trains **both** variants sequentially (one GPU) in a **background process**, so a
dropped notebook connection doesn't kill the run:

- per variant: seed a fresh exp dir with the released weights → `train.py`
  (`-d strudel_ft_<variant>`, arch flags, bf16) → copy checkpoint + log +
  `metadata.json` to that variant's Drive folder;
- **OOM auto-retry**: batch (4,8) → (2,4) → (1,2);
- progress is written to `COMPARE_DIR/training_status.json` — re-run the polling
  cell (7c) anytime. Keep the Colab tab open; each variant takes hours.


In [ ]:
# 7a. ensure the ~1GB released checkpoint (Drive-cached). Seeding per variant is
# done by the driver in 7b.
RELEASED_CKPT = MODEL_ROOT / "amt" / "logs" / "2024" / RELEASED_EXP / "checkpoints" / "last.ckpt"

def ensure_checkpoint():
    if RELEASED_CKPT.exists():
        print("checkpoint already local"); return
    cache_ck = DRIVE_MODEL_CACHE / "amt" / "logs" / "2024" / RELEASED_EXP / "checkpoints" / "last.ckpt"
    if cache_ck.exists():
        print("restoring checkpoint from Drive cache…")
        RELEASED_CKPT.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cache_ck, RELEASED_CKPT)
    else:
        print("fetching checkpoint from HF model repo (~1GB, one-time)…")
        snapshot_download(repo_id="mimbres/YourMT3", repo_type="model",
                          allow_patterns=[f"logs/2024/{RELEASED_EXP}/**"], local_dir=str(MODEL_ROOT / "amt"))
        cache_ck.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(RELEASED_CKPT, cache_ck)
    assert RELEASED_CKPT.exists(), "checkpoint missing after ensure_checkpoint()"

ensure_checkpoint()
print("released checkpoint ready:", RELEASED_CKPT.exists())

# Effective sampling mix per variant. YourMT3's WeightedRandomSampler gives each
# file weight W_D * (1 - n_D/N), so the realized mix differs from the nominal one.
counts = {"strudel": sum(len(DATA["train"][c]["x"]) for c in STRUDEL_SUBCATS),
          "slakh": len(DATA["train"]["slakh"]["x"]),
          "maestro": len(DATA["train"]["maestro"]["x"]),
          "egmd": len(DATA["train"]["egmd"]["x"])}
N = sum(counts.values())
print(f"\ntrain pool: {counts}  (N={N:,})")
for vname, w in WEIGHT_VARIANTS.items():
    eff = {d: counts[d] * w[d] * (1 - counts[d] / N) for d in counts}
    s = sum(eff.values()) or 1                       # guard: reference sets absent locally
    print(f"{vname:<10} nominal " + "/".join(f"{d}:{w[d]:.2f}" for d in counts)
          + "  ->  effective " + "  ".join(f"{d} {100 * v / s:4.1f}%" for d, v in eff.items()))


In [ ]:
# 7b. launch the background driver: trains BOTH variants sequentially, with OOM
# retry, saving everything per variant to Drive. Survives notebook disconnects.
assert torch and torch.cuda.is_available(), "no GPU — set Runtime > Change runtime type > GPU"
STATUS = COMPARE_DIR / "training_status.json"

DRIVER_CFG = COMPARE_DIR / "driver_config.json"
DRIVER_CFG.write_text(json.dumps({
    "model_root": str(MODEL_ROOT), "released_ckpt": str(RELEASED_CKPT), "status": str(STATUS),
    "arch": ARCH, "precision": PRECISION, "lr": LR, "max_steps": MAX_STEPS,
    "bsz_ladder": [[4, 8], [2, 4], [1, 2]] if not SMOKE_TEST else [[1, 1]],
    "variants": {v: {"preset": f"strudel_ft_{v}", "exp": VARIANT_EXP[v],
                     "run_dir": str(VARIANT_RUN_DIR[v]), "weights": dict(WEIGHT_VARIANTS[v])}
                 for v in WEIGHT_VARIANTS}}, indent=2))

DRIVER = (Path("/content") if IN_COLAB else REPO) / f"driver_{TIMESTAMP}.py"
DRIVER.write_text(r"""
import json, os, shutil, subprocess, sys, time, datetime
from pathlib import Path
cfg = json.loads(Path(sys.argv[1]).read_text())
MODEL_ROOT = Path(cfg["model_root"]); RELEASED = Path(cfg["released_ckpt"]); STATUS = Path(cfg["status"])
env = dict(os.environ); env.update({"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True", "TORCHDYNAMO_DISABLE": "1"})
state = {"started": datetime.datetime.now().isoformat(timespec="seconds"), "variants": {}}
def save(): STATUS.write_text(json.dumps(state, indent=2))
save()
for vname, v in cfg["variants"].items():
    exp = v["exp"]; run_dir = Path(v["run_dir"]); run_dir.mkdir(parents=True, exist_ok=True)
    ck = MODEL_ROOT / "amt" / "logs" / "ymt3" / exp / "checkpoints" / "last.ckpt"
    ck.parent.mkdir(parents=True, exist_ok=True)
    if not ck.exists():
        shutil.copy2(RELEASED, ck)          # seed with released weights
    log = MODEL_ROOT / "amt" / "logs" / "ymt3" / exp / "train_stdout.log"
    st = {"preset": v["preset"], "exp": exp, "status": "starting", "bsz": None, "t0": time.time()}
    state["variants"][vname] = st; save()
    ok = False
    for bsz in cfg["bsz_ladder"]:
        st["bsz"] = bsz; st["status"] = "running bsz=%s" % bsz; save()
        cmd = [sys.executable, "amt/src/train.py", exp, "-d", v["preset"], *cfg["arch"],
               "-pr", cfg["precision"], "-bsz", str(bsz[0]), str(bsz[1]),
               "-lr", str(cfg["lr"]), "-it", str(cfg["max_steps"]), "-g", "auto"]
        with open(log, "w") as f:
            r = subprocess.run(cmd, cwd=MODEL_ROOT, stdout=f, stderr=subprocess.STDOUT, env=env)
        if r.returncode == 0:
            ok = True; break
        if "OutOfMemoryError" not in log.read_text()[-6000:]:
            break                            # non-OOM failure: don't retry blindly
    st["wall_sec"] = round(time.time() - st["t0"]); st["status"] = "done" if ok else "FAILED"; save()
    for c in sorted(ck.parent.glob("*.ckpt")):
        shutil.copy2(c, run_dir / c.name)
    if log.exists():
        shutil.copy2(log, run_dir / "train_stdout.log")
    (run_dir / "metadata.json").write_text(json.dumps({
        "variant": vname, "weights": v["weights"], "max_steps": cfg["max_steps"], "lr": cfg["lr"],
        "precision": cfg["precision"], **st}, indent=2))
    if not ok:
        break
state["finished"] = datetime.datetime.now().isoformat(timespec="seconds"); save()
""")
proc = subprocess.Popen([sys.executable, str(DRIVER), str(DRIVER_CFG)],
                        stdout=open(COMPARE_DIR / "driver_stdout.log", "w"), stderr=subprocess.STDOUT)
print(f"driver started (pid {proc.pid}) — trains {list(WEIGHT_VARIANTS)} sequentially")
print("status file:", STATUS)
print("poll with cell 7c; KEEP THIS COLAB TAB OPEN (each variant takes hours)")


In [ ]:
# 7c. poll training progress (re-run this cell anytime)
STATUS = COMPARE_DIR / "training_status.json"
if not STATUS.exists():
    print("no status yet — did 7b run?")
else:
    st = json.loads(STATUS.read_text())
    print(json.dumps({k: v for k, v in st.items() if k != "variants"}, indent=2))
    for vname, v in st.get("variants", {}).items():
        extra = f"  wall={v.get('wall_sec', '?')}s" if "wall_sec" in v else ""
        print(f"\n[{vname}] {v['status']}  bsz={v.get('bsz')}{extra}")
        log = MODEL_ROOT / "amt" / "logs" / "ymt3" / v["exp"] / "train_stdout.log"
        if log.exists():
            tail = [l for l in log.read_text().splitlines() if l.strip()][-2:]
            print("   " + "\n   ".join(tail))
    if "finished" in st:
        print("\nALL TRAINING FINISHED — continue with §8 (curves) and §9 (benchmark).")


## 8. Training curves

Train/validation loss for **both variants**, from the `CSVLogger` patched in §3.
Run after the driver reports `finished` (7c).


In [ ]:
def series(rows, *keys):
    for key in keys:                                   # PL suffixes: train_loss_step etc.
        xs, ys = [], []
        for r in rows:
            v = r.get(key, "")
            if v not in ("", None):
                xs.append(float(r.get("step") or r.get("epoch") or len(xs))); ys.append(float(v))
        if xs:
            return xs, ys
    return [], []

fig, axes = plt.subplots(1, len(VARIANT_EXP), figsize=(8 * len(VARIANT_EXP), 5), squeeze=False)
for ax, (vname, exp) in zip(axes[0], VARIANT_EXP.items()):
    mfiles = sorted((MODEL_ROOT / "amt" / "logs" / "ymt3" / exp / "metrics").glob("version_*/metrics.csv"))
    if not mfiles:
        ax.set_title(f"{vname} — no metrics.csv yet"); continue
    rows = list(csv.DictReader(open(mfiles[-1])))
    for keys, lab, col in ((("train_loss", "train_loss_step"), "train loss", "#d1495b"),
                           (("val_loss", "val_loss_epoch"), "val loss", "#2e4057")):
        xs, ys = series(rows, *keys)
        if xs:
            ax.plot(xs, ys, marker="o", ms=2, lw=1, label=f"{lab} (n={len(xs)})", color=col)
    ax.set_xlabel("step"); ax.set_ylabel("loss"); ax.grid(alpha=0.3); ax.legend()
    ax.set_title(f"{vname} — {dict(WEIGHT_VARIANTS[vname])}", fontsize=9)
fig.suptitle("Fine-tuning loss — both weight variants", y=1.02)
plt.tight_layout(); fig.savefig(COMPARE_DIR / "loss_curves.png", dpi=120); plt.show()
print("saved ->", COMPARE_DIR / "loss_curves.png")


## 9. Benchmark — base vs. both fine-tunes, per category AND per instrument group

Runs YourMT3's `test.py` for **3 models** (released base + both variants) on
**each dataset category's** test split (capped by `EVAL_CAP` unless `FULL_EVAL`).
Full metric names (incl. per-instrument-class F1) are read from the result files
`test.py` writes, then aggregated two ways:

1. **per dataset category** — strudel_corpus / maestro / slakh / egmd;
2. **per instrument group** on the target (strudel_corpus): **drums, synth
   leads, bass, chords (pads/keys)**.

> `strudel_synthetic_batch_*` have **no test split by design** (scoring our own
> synthetic data would flatter results). And **"effects"** (`lpf`/`room`/`delay`)
> aren't note events — no note-F1 class exists for them; they influence the
> overall scores only indirectly through timbre.


In [ ]:
# 3 models x all categories with a test split. Results parsed from test.py's
# result files (full metric names incl. per-instrument classes).
AVAILABLE = [c for c in CATEGORIES if BENCH_N.get(c, 0) > 0]
BENCH_CATS = AVAILABLE if not SMOKE_TEST else [c for c in ("strudel_corpus", "maestro") if c in AVAILABLE]
MODELS = OrderedDict([("base", (RELEASED_EXP, "2024"))])
for vname, exp in VARIANT_EXP.items():
    MODELS[vname] = (exp, "ymt3")
print("models:", list(MODELS), "| categories:", BENCH_CATS)

BENCH_LOGDIR = COMPARE_DIR / "bench_logs"; BENCH_LOGDIR.mkdir(parents=True, exist_ok=True)

def run_test(mname, exp_id, project, cat):
    exp_dir = MODEL_ROOT / "amt" / "logs" / project / exp_id
    cmd = [sys.executable, "amt/src/test.py", exp_id, "-p", project, "-d", f"bench_{cat}",
           *ARCH, "-pr", PRECISION, "-g", "auto"]
    log = exp_dir / f"bench_{cat}_stdout.log"
    with open(log, "w") as f:
        subprocess.run(cmd, cwd=MODEL_ROOT, stdout=f, stderr=subprocess.STDOUT)
    shutil.copy2(log, BENCH_LOGDIR / f"{mname}_{cat}.log")
    # full metric names live in the result file test.py writes (pprint format)
    m = {}
    for rf in sorted(exp_dir.glob(f"result_*_bench_{cat}.json")):
        for name, val in re.findall(r"'test/\([^)]*\)([^']+)':\s*(?:tensor\()?([0-9.]+)", rf.read_text()):
            m[name] = float(val)
        shutil.copy2(rf, BENCH_LOGDIR / f"{mname}_{cat}_result.json")
    if not m:  # fallback: parse the Rich table from stdout (truncated names)
        for name, val in re.findall(r"test/\([^)]*\)(\w+)\s*│\s*([\d.]+)", log.read_text()):
            m[name] = float(val)
    return m

BENCH = {}
for cat in BENCH_CATS:
    BENCH[cat] = {}
    for mname, (exp_id, project) in MODELS.items():
        BENCH[cat][mname] = run_test(mname, exp_id, project, cat)
        print(f"{cat:<18} {mname:<12} {len(BENCH[cat][mname])} metrics")
print("\nexample strudel_corpus/base keys:", sorted(BENCH.get("strudel_corpus", {}).get("base", {}))[:12])


In [ ]:
# Two comparisons: per dataset category, and per instrument group on the target.
MODEL_COLORS = {"base": "#8d99ae", "strudel50": "#d1495b", "egmd50": "#2e4057"}

def primary(metrics):
    for pref in ("onset_f", "multi_f", "frame_f"):
        if pref in metrics:
            return metrics[pref]
    return next(iter(metrics.values()), float("nan"))

# --- (1) per dataset category -------------------------------------------------
cats = [c for c in BENCH]
print(f"{'category':<18}" + "".join(f"{m:>12}" for m in MODELS))
for c in cats:
    print(f"{c:<18}" + "".join(f"{primary(BENCH[c][m]):>12.3f}" for m in MODELS))

x = np.arange(len(cats)); w = 0.8 / len(MODELS)
fig, ax = plt.subplots(figsize=(max(9, 2 * len(cats)), 5))
for j, m in enumerate(MODELS):
    ax.bar(x + (j - (len(MODELS) - 1) / 2) * w, [primary(BENCH[c][m]) for c in cats], w,
           label=m, color=MODEL_COLORS.get(m))
ax.set_xticks(x); ax.set_xticklabels(cats, rotation=20, ha="right", fontsize=8)
ax.set_ylabel("note F1 (primary)"); ax.legend(); ax.grid(axis="y", alpha=0.3)
ax.set_title("All models × dataset categories")
plt.tight_layout(); fig.savefig(COMPARE_DIR / "comparison_categories.png", dpi=120); plt.show()

# --- (2) per instrument group on the TARGET (strudel_corpus) -------------------
# groups matched by substring against the full per-class metric names (onset_f_<Class>);
# 'effects' has no note class (FX change timbre, not notes) — shown as n/a.
GROUPS = OrderedDict([
    ("drums",           ["drum"]),
    ("synth leads",     ["Synth Lead"]),
    ("bass",            ["Bass"]),
    ("chords (pads/keys)", ["Synth Pad", "Piano", "Organ"]),
    ("effects",         []),
])
def group_score(metrics, pats):
    vals = [v for k, v in metrics.items()
            if k.startswith("onset_f") and any(p.lower() in k.lower() for p in pats)]
    return float(np.mean(vals)) if vals else float("nan")

tgt = "strudel_corpus"
if tgt in BENCH:
    print(f"\nInstrument groups on {tgt} (mean onset-F1 over matching classes):")
    print(f"{'group':<20}" + "".join(f"{m:>12}" for m in MODELS))
    gvals = {m: [group_score(BENCH[tgt][m], pats) for pats in GROUPS.values()] for m in MODELS}
    for i, g in enumerate(GROUPS):
        print(f"{g:<20}" + "".join(f"{gvals[m][i]:>12.3f}" for m in MODELS))
    x = np.arange(len(GROUPS))
    fig, ax = plt.subplots(figsize=(10, 5))
    for j, m in enumerate(MODELS):
        ax.bar(x + (j - (len(MODELS) - 1) / 2) * w, gvals[m], w, label=m, color=MODEL_COLORS.get(m))
    ax.set_xticks(x); ax.set_xticklabels(list(GROUPS), rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("mean onset F1"); ax.legend(); ax.grid(axis="y", alpha=0.3)
    ax.set_title(f"Instrument groups on {tgt} — 'effects' has no note class (n/a)")
    plt.tight_layout(); fig.savefig(COMPARE_DIR / "comparison_instrument_groups.png", dpi=120); plt.show()

# --- persist EVERYTHING for later custom viz ----------------------------------
results = {
    "run": RUN_NAME, "created": datetime.datetime.now().isoformat(timespec="seconds"),
    "models": {m: {"exp": MODELS[m][0], "project": MODELS[m][1],
                   "weights": dict(WEIGHT_VARIANTS[m]) if m in WEIGHT_VARIANTS else None} for m in MODELS},
    "eval_cap": EVAL_CAP, "full_eval": FULL_EVAL, "n_test_files": {c: BENCH_N.get(c) for c in cats},
    "metrics": BENCH,
    "instrument_groups": {g: pats for g, pats in GROUPS.items()},
}
(COMPARE_DIR / "comparison_results.json").write_text(json.dumps(results, indent=2, default=str))
print("saved ->", COMPARE_DIR / "comparison_results.json")

print(f"\n=== all artifacts in Google Drive: {COMPARE_DIR} ===")
for f in sorted(COMPARE_DIR.rglob("*")):
    if f.is_file():
        print(f"  {str(f.relative_to(COMPARE_DIR)):<40} {f.stat().st_size/1e6:8.2f} MB")
